# NADOR quickstart: what monitoring can do

NADOR adds monitoring to a 5G network deployed with IFRAN and records experiments on it. On one
small network (one AMF, SMF, UPF and gNB) this notebook shows each capability in turn:

| Capability | Section |
| --- | --- |
| Describe what to collect, by where it measures (RAN, core, end to end, hosts), with every mistake caught before deployment | 2. The monitoring profile |
| See exactly what would be installed, without touching the cluster | 3. Plan the deployment |
| Deploy Prometheus, Grafana, Loki, the controller, the AMF/SMF sniffers and the probes in one call | 4. Deploy monitoring |
| Record an experiment, with iperf traffic, and get metrics, logs, UE context and host snapshots saved per section | 5. Run an experiment, 6. Inspect what was saved |
| Add your own evidence to every run | 7. Collect your own evidence |
| Start, stop, rescope, deploy or remove probes on the running network | 8. Control probes live |
| Check that every metric actually arrives | 9. Check that metrics arrive |
| Read the monitoring components' logs, save and restore the handle, remove everything | 10 to 12 |

Every metric and experiment option is listed in the [Reference](#reference) at the end.

Download this notebook, [monitoring.yaml](monitoring.yaml) and [queries.json](queries.json)
into one folder and use a Python 3.12+ kernel.

Sections 1 and 2 run anywhere. From section 3 on you need the `ran` manager of a deployed
core and RAN, with current inventory and SSH access (see the [RFsim examples](../monitoring/rfsim)
for a full network setup). Keep UEs unattached until monitoring is deployed.

## 1. Install and import

Install once per Python environment, then restart the kernel.

In [ ]:
%pip install --upgrade nador

In [ ]:
import asyncio
import csv
import gzip
import json
import subprocess
from datetime import UTC, datetime
from pathlib import Path

from nador import Metrics, MonitoringDeployment, MonitoringManagement, configure_ran_kpis
from nador.artifacts import Collector, ExperimentRun, iperf_summary, prometheus_connection
from nador.config import validate_deployment

## 2. The monitoring profile

[monitoring.yaml](monitoring.yaml) says what to collect and where:

- `metrics:` lists **every** metric NADOR knows, grouped by where it measures: `ran`, `core`,
  `e2e` (end to end, between RAN and core) and `host`. `false` leaves a metric out. A metric without
  options is turned on with `true`; one with options is turned on by writing all of them, defaults
  included. So a profile always says exactly what it collects and how.
- `targets:` are the network functions to watch. A `role` (`amf`, `smf`, `upf`, `gnb`) marks where
  packets are captured; `container` and `interface` say which container and interface.
- `images:` (optional) overrides container images. Every default is pinned in NADOR's `images.json`.
- `tuning:` (optional) holds rarely changed knobs such as probe intervals.

Namespaces and pod selectors come from `ran`, so the profile needs no credentials.

In [ ]:
config = MonitoringDeployment.from_yaml_file("monitoring.yaml")

for name in Metrics.model_fields:
    state = "on" if config.metrics.running(name) else "off"
    print(f"{name:22} {state}")

### Metric options

A metric with options lists all of them. `rtt`, for example, chooses where it measures (`at`:
the gNB, the UPF or both), whether to pair packets seen at both (`packet_pairing`, which needs
both), and which UEs and slices (empty lists mean all). Metrics the controller manages also take
`on_demand: true`: they are deployed stopped and started later (section 8).

```yaml
metrics:
  core:
    ngap: {port: 9102}                               # AMF session-setup metrics
    pfcp: {port: 9104}                               # SMF session-establishment metrics
  e2e:
    rtt: {on_demand: false, at: [upf], packet_pairing: false, imsis: [], slices: ["01:ffffff"]}
  host:
    logs: true                                       # logs of NADOR's monitoring pods
    scheduler_wait: {on_demand: true, targets: []}   # CPU scheduler probe, started later
tuning:
  probe_interval_seconds: 10
```

(Only the changed metrics are shown; a real profile lists all of them.) Leave an option out, or
put a metric in the wrong group, and the error says what to write. The cell below builds the
same variant from the loaded profile, so you can try options without editing the file;
`explicit=True` applies the same checks as reading a file.

In [ ]:
data = config.model_dump()
data["metrics"]["core"] |= {"ngap": {"port": 9102}, "pfcp": {"port": 9104}}
data["metrics"]["e2e"]["rtt"] = {
    "on_demand": False, "at": ["upf"], "packet_pairing": False, "imsis": [], "slices": ["01:ffffff"]
}
data["metrics"]["host"] |= {"logs": True, "scheduler_wait": {"on_demand": True, "targets": []}}
data["tuning"] |= {"probe_interval_seconds": 10}
richer = validate_deployment(data, explicit=True)

print("running:  ", [name for name in Metrics.model_fields if richer.metrics.running(name)])
print("on demand:", [name for name in Metrics.model_fields if richer.metrics.available(name) and not richer.metrics.running(name)])
print("rtt at:", richer.metrics.rtt.at, "slices:", richer.metrics.rtt.slices)

data["metrics"]["e2e"]["rtt"] = True
try:
    validate_deployment(data, explicit=True)
except ValueError as error:
    print(error)

Every metric, its options and defaults, and the `tuning:` knobs are listed in the
[Reference](#reference) at the end.

### Images

Every image NADOR deploys has a key in `images.json`. Override one by key, for example a
custom RTT probe build. Images must be pinned to a tag or digest.

In [ ]:
print("default RTT probe:", config.image("rtt_probe"))

data = config.model_dump()
data["images"] = {"rtt_probe": "ghcr.io/your-org/ebpf-latency-probe:v2"}
custom = validate_deployment(data)
print("custom RTT probe: ", custom.image("rtt_probe"))

### Mistakes are caught before anything is deployed

NADOR checks that each metric has what it needs. Here `upf_transit` is turned on without its
test peer, and a probe image has no tag:

In [ ]:
data = config.model_dump()
data["metrics"] |= {"upf_transit": True}
data["images"] = {"controller": "ghcr.io/your-org/nador-controller"}

try:
    validate_deployment(data)
except ValueError as error:
    print(error)

Pick the profile to deploy. This notebook uses the file as is; use `richer` to deploy the
variant built above.

In [ ]:
profile = config  # or: richer

## 3. Plan the deployment

Run your network setup first so that `ran` exists. `plan()` locates every target in the deployed
network and renders all Helm values **without touching the cluster**, so you can check what will be
installed.

In [ ]:
monitoring = MonitoringManagement(preceding=ran, config=profile)
plan = monitoring.plan()

print("control node:", plan.control_node)
for target in plan.targets:
    print(f"target {target.component:6} role={target.role} namespace={target.namespace} {target.label_selectors}")
print("Helm releases:", list(plan.components))

**OCUDU RANs only.** To collect OCUDU's native KPIs, turn on `ran_kpis` and enable the RAN's JSON
metrics **before** deploying the RAN:

```python
configure_ran_kpis(ran, profile)  # then: await ran.deploy()
```

OAI RANs need no extra step.

## 4. Deploy monitoring, then attach UEs

`deploy()` installs Prometheus, Grafana (`admin` / `admin`), the probes and, when a metric needs
UE context, the UE mapper and the AMF/SMF sniffers. Deploy before attaching UEs, so the sniffers
see session establishment.

In [ ]:
await monitoring.deploy()

Then attach the UEs. For containerized UEs managed by IFRAN the usual sequence is below; physical
UEs connect with their normal procedure.

In [ ]:
from ifran.deployment import UEManagement

ue = UEManagement(preceding=ran)
await ue.prepare()
await ue.install()

## 5. Run an experiment

`ExperimentRun` records when each `section` of your workload ran and, with the monitoring manager,
saves everything the deployment observes meanwhile:

- metrics from Prometheus, for the queries in `queries.json`, split per section;
- logs from Loki, for the groups whose `logs` is on (RAN, core, monitoring pods);
- the UE context over time (UE, IP, slice, tunnels), when the UE mapper is deployed;
- snapshots of the hosts and the cluster before and after the run.

Choose a UE component, its tunnel interface and an address reachable through its 5G connection.
The UE container must provide `ping`.

In [ ]:
UE_COMPONENT = "sim_ue"
UE_INTERFACE = "oaitun_ue1"
TARGET_IP = "<address reachable through the UE tunnel>"

result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-ping"

async with ExperimentRun(result_dir, name="ue-ping", monitoring=monitoring, queries_file="queries.json") as run:
    with run.section("idle"):
        await asyncio.sleep(5)  # a baseline to compare against

    with run.section("ping"):
        report = await ue.run_command(
            command=["ping", "-I", UE_INTERFACE, "-c", "10", TARGET_IP],
            components=[UE_COMPONENT],
            quiet=True,
            flatten=False,
        )
    run.save_json("ping.json", report)  # your own results go in the same folder

    await asyncio.sleep(2)  # let Prometheus scrape the last samples

print(f"Saved to {result_dir.resolve()}")

### Throughput with iperf3

`Iperf.setup` installs iperf3 on a server node and on the UEs that lack it, and finds each UE's
address on its tunnel interface. `iperf.run(ue, direction, seconds)` starts a one-shot server on a
fresh port and runs the UE side; it returns iperf3's JSON report, with an `error` key if the test
failed. With physical UEs, pass a `ProvisioningManagement` instead of `ue`.

In [ ]:
from nador.traffic import Iperf

SERVER_NODE = "<node with a route to the UE subnet, e.g. the UPF's node>"

iperf = await Iperf.setup(ue, server=SERVER_NODE, interface=UE_INTERFACE, clients=[UE_COMPONENT])
result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-iperf"

async with ExperimentRun(result_dir, name="ue-iperf", monitoring=monitoring, queries_file="queries.json") as run:
    for direction in ("downlink", "uplink"):
        with run.section(direction):
            report = await iperf.run(UE_COMPONENT, direction, 10)
        run.save_json(f"{direction}.json", report)
        if "error" in report:
            raise RuntimeError(report["error"])
        print(direction, f"{report['end']['sum_received']['bits_per_second'] / 1e6:.1f} Mbit/s")
    await asyncio.sleep(2)
print(iperf_summary(result_dir))

If a section raises, the run is recorded as `failed` and the exception reaches you as usual;
evidence is still collected. If collection itself has a problem on a successful run, the run raises
`RuntimeError` and `experiment_metadata.json` lists what went wrong.

### Narrowing what is collected

Keyword options tune the run. Logs are the usual thing to narrow: by default every line Loki holds
is exported, which is the namespaces whose `logs` the profile turned on. `loki_selector` takes a
Loki label selector (LogQL), so you can keep only
some namespaces or pods, and `loki_limit` caps the lines per query. This run also skips the host
snapshots and samples the UE context every second:

In [ ]:
result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-narrow"

async with ExperimentRun(
    result_dir,
    monitoring=monitoring,
    loki_selector='{namespace=~"core|ran"}',  # or one pod: '{namespace="core", pod=~"amf.*"}'
    loki_limit=2000,
    collect_setup=False,
    interval_seconds=1,
) as run:
    with run.section("ping"):
        await ue.run_command(
            command=["ping", "-I", UE_INTERFACE, "-c", "10", TARGET_IP], components=[UE_COMPONENT], quiet=True
        )

print(json.loads((result_dir / "logs" / "summary.json").read_text()))

Logs are only exported when `logs` is on in the profile; `summary.json` says how many lines were
exported and whether the export may be incomplete.

Every `ExperimentRun` option and its default is listed in the [Reference](#reference) at the end.

## 6. Inspect what was saved

In [ ]:
for path in sorted(result_dir.rglob("*")):
    if path.is_file():
        print(f"{path.stat().st_size:>10}  {path.relative_to(result_dir)}")

- `experiment_metadata.json`: status (`completed`, `failed` or `cancelled`) and any collection errors.
- `timeline_summary.csv`, `timeline_recap.txt`: when the campaign and each section ran.
- `prometheus_timeseries.csv`: every sample of every query; `prometheus_export_summary.json` says
  how many rows each query returned (an empty query had no samples, which is not an error).
- `by_window/section/<name>/`: the same samples cut to one section.
- `ue_context/`: sampled UE inventory; `setup/before|after/`: host and cluster snapshots;
  `logs/`: Loki export, when logs are on.

In [ ]:
metadata = json.loads((result_dir / "experiment_metadata.json").read_text())
print("status:", metadata["status"], "| collection errors:", metadata["collection_errors"])

print((result_dir / "timeline_recap.txt").read_text())

with gzip.open(result_dir / "by_window/section/section__ping/prometheus_timeseries.csv.gz", "rt") as f:
    rows = list(csv.DictReader(f))
print(f"{len(rows)} samples during 'ping', queries: {sorted({row['query_name'] for row in rows})}")

## 7. Collect your own evidence

A collector is any object with up to three steps, all optional:

- `start(run)`: before the workload,
- `stop(run)`: after the workload (in reverse order),
- `collect(run, start, end)`: after the timeline is written, for the run's time range.

If a step fails, the error is recorded and the other collectors still run. A collector can also
record a problem without raising, with `run.error(label, problem)`, and write files anywhere in
the run folder with `run.save_json(path, data)`, like your own results. This one saves the UE's addresses before and
after the workload:

In [ ]:
class UeAddresses(Collector):
    label = "ue_addresses"

    async def _save(self, run, phase):
        output = await ue.run_command(["ip", "-j", "address"], components=[UE_COMPONENT], quiet=True, flatten=False)
        run.save_json(f"ue_addresses/{phase}.json", output)

    async def start(self, run):
        await self._save(run, "before")

    async def stop(self, run):
        await self._save(run, "after")


result_dir = Path("results") / f"{datetime.now(UTC):%Y%m%dT%H%M%S}-custom"
async with ExperimentRun(result_dir, monitoring=monitoring, collectors=[UeAddresses()]) as run:
    with run.section("wait"):
        await asyncio.sleep(5)

print(sorted(path.name for path in (result_dir / "ue_addresses").iterdir()))

## 8. Control probes live

Metrics the controller manages (`rtt`, `throughput`, `n3_one_way`, `upf_transit`,
`user_plane_paths` and the host probes) can be started and stopped without redeploying.
Deploy them with `on_demand: true` to have them start stopped. `start` also changes the scope of
a running metric: `rtt` and `throughput` take `imsis` or `slices` to measure only some UEs (an
empty list measures all of them again), and `rtt` takes `at` to move between gNB and UPF.

Host-probe metrics start and stop with their probe: `kernel_drops` and `attributed_drops` share
one probe, as do `socket_counters` and `tcp_flows`. The controller applies a change within a few
seconds.

In [ ]:
await monitoring.start("rtt", slices=["01:ffffff"], at=["upf"])  # one slice, at the UPF only
await monitoring.start("scheduler_wait")
await monitoring.start("rtt", slices=[], at=["gnb", "upf"])      # every UE again, both anchors
await monitoring.stop("scheduler_wait")

`stop` keeps a probe installed and only removes its running pods. To add a metric that is not
deployed, or to remove one completely, use `deploy_probe` and `remove_probe`. Options are
written as in `monitoring.yaml`. Only the releases the change affects are installed, upgraded or
uninstalled, and the AMF/SMF sniffers start with the first metric that needs them (UEs attached
before that are seen once they reattach).

In [ ]:
await monitoring.deploy_probe("kernel_drops", {"on_demand": False, "targets": []})
await monitoring.remove_probe("kernel_drops")

## 9. Check that metrics arrive

A finished experiment does not prove every metric had data. `nador-check-metrics` runs every
query against Prometheus and reports which ones returned samples, and which scrape targets are down.

In [ ]:
report = Path("results") / "metrics-check.json"
async with prometheus_connection(monitoring) as url:
    subprocess.run(["nador-check-metrics", "--prometheus-url", url, "--out", str(report)], check=False)
print(json.loads(report.read_text())["counts"])

## 10. Read the monitoring components' logs

In [ ]:
print(await monitoring.get_logs(components=["ue-mapper-api"], since_seconds=600, quiet=True))

## 11. Save and restore the deployment handle

Save the models after deploying. After a kernel restart, restore the handle without redeploying.
Keep these files private: they contain host access details.

In [ ]:
monitoring.output(metamodel_file="monitoring_metamodel.yaml", deployment_model_file="monitoring_deployment_model.yaml")

# After a restart:
monitoring = MonitoringManagement.from_yaml_files("monitoring_metamodel.yaml", "monitoring_deployment_model.yaml")

## 12. Remove monitoring

This removes monitoring only; the core, RAN, UEs and saved results stay.

In [ ]:
# await monitoring.remove()

## Reference

The details behind sections 2 and 5, to come back to when writing your own profile or run.

### Metrics, options and tuning

| Group | Metric | What it collects | Options (default) |
| --- | --- | --- | --- |
| `ran` | `ran_kpis` | OAI or OCUDU RAN KPIs (call `configure_ran_kpis` before deploying an OCUDU RAN) | `ocudu_port` (8001) |
| `ran` | `csi` | Uplink channel per UE from the OAI or OCUDU gNB/DU (call `configure_csi` before deploying the RAN) | `components`, `srs_period_ms`, `granularity`, ... |
| `ran` | `split_signaling` | F1/E1 signalling between CU and DU (split RANs only) | `targets` |
| `ran` | `logs` | Logs of the RAN's pods, in Loki | none |
| `core` | `ngap` | Session-setup times, from the AMF sniffer | `port` (9102) |
| `core` | `pfcp` | Session-establishment times, from the SMF sniffer | `port` (9104) |
| `core` | `throughput` | N3 throughput per UE and slice, at the UPFs | none besides `on_demand` |
| `core` | `upf_transit` | One-way latency through the UPF towards a test peer | `peer`, `test_port` (required) |
| `core` | `ue_mapper_coverage` | How many UEs the mapper resolved to IMSI, IP and tunnels | none |
| `core` | `logs` | Logs of the core's pods, in Loki | none |
| `e2e` | `rtt` | TCP round-trip time per UE and slice, on N3 | `at` ([gnb, upf]), `packet_pairing` (false, needs both), `imsis`/`slices` ([] = all) |
| `e2e` | `n3_one_way` | One-way gNB-UPF latency on N3 | `log_samples` (false): also log every accepted sample as JSON |
| `e2e` | `user_plane_paths` | Packet counters per user-plane path on gNBs and UPFs | `port` (9403), `interval_seconds` (5) |
| `host` | `node_metrics` | Node CPU, memory, disk and network (node exporter) | none |
| `host` | `logs` | Logs of NADOR's monitoring pods (probes, controller, mapper), in Loki | none |
| `host` | `scheduler_wait` | How long tasks wait for a CPU on the nodes | `targets` ([] = all) |
| `host` | `kernel_drops`, `attributed_drops` | Packets the kernel drops, and the flows they belonged to | `targets` |
| `host` | `network_queues` | Queueing in network interface queues | `targets` |
| `host` | `socket_counters`, `tcp_flows` | Socket statistics, and per-flow TCP evidence | `targets`; `tcp_flows` also `identity_secret` (required) |
| `host` | `measurement_quality` | Freshness and availability of chosen Prometheus queries | `targets`, `queries` ([]) |

The controller-managed metrics (`throughput`, `upf_transit`, the `e2e` group, `split_signaling` and
the host probes from `scheduler_wait` down) also take `on_demand` (false). `targets` limits a host
probe to some components, such as `ran/gnb`. Host probes that share a row run in one probe. `attributed_drops` and `tcp_flows` only collect when
deployed running.

`tuning:` holds knobs shared by several probes; the defaults suit most deployments:
`probe_interval_seconds` (5), `probe_timeout_seconds` (2), `probe_capture_limit` (8),
`packet_run_id` ("experiment"), `packet_state_limit` (4096), `packet_ttl_seconds` (30),
`sniffer_pending_timeout_seconds` (30), `one_way_sample` (16).

### Experiment options

`ExperimentRun(directory, ...)` takes these keywords; the defaults suit most runs.

| Option | Default | What it does |
| --- | --- | --- |
| `name` | "experiment" | Name recorded in `experiment_metadata.json` |
| `monitoring` | none | The monitoring manager: everything it deployed is saved, through SSH tunnels |
| `prometheus_url` | none | A directly reachable Prometheus instead of `monitoring` (metrics only) |
| `queries_file` | NADOR's `queries.json` | The Prometheus queries to export |
| `step` | "1s" | Resolution of the exported time series |
| `timezone` | "UTC" | Time zone of the readable timestamps in the timeline and its recap |
| `collectors` | () | Your own collectors (section 7), run alongside the built-in ones |
| `collect_setup` | True | Snapshot the hosts and cluster before and after |
| `setup_hosts` | every reachable node | Only snapshot these nodes |
| `collect_ue_context` | True | Sample the UE context (UE, IP, slice, tunnels) during the run |
| `mapper_url` | deployed mapper | A UE mapper reachable directly, instead of through the cluster |
| `namespace`, `service`, `service_port` | monitoring namespace, "ue-mapper-api", 8080 | Where the UE mapper runs |
| `interval_seconds` | 2 | Seconds between UE context samples |
| `timeout_seconds` | 5 | Timeout of one UE context request |
| `max_seconds` | 7200 | Longest run the UE sampler follows |
| `limit` | 5000 | Most UEs per sample |
| `loki_url` | deployed Loki | A Loki reachable directly; also turns on log export without `monitoring` |
| `loki_selector` | everything collected | Which logs to export (LogQL selector) |
| `loki_limit` | 5000 | Most log lines per query |
| `loki_max_queries` | 1024 | Most Loki queries per run |

Pass neither `monitoring` nor `prometheus_url` to record only the timeline.